# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuguda999/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

**The decay/refresh insight this playbook operationalizes:** a page is worth reviewing first when it already has real demand (impressions) AND shows a sign of losing it — not just because it's declining (thin, unread pages decline constantly and nobody should care) and not just because it's popular (a popular stable page needs nothing). Demand × risk is what earns a human's limited time. This has been the throughline since ML-02.

**Five archetypes**, each a reason code with its own action — built entirely from artifacts already validated in ML-07/08/09, no new unvalidated clustering:

| Archetype (reason code) | What it means | Action | Rank key |
|---|---|---|---|
| `low_ctr_for_position` | ML-07's CONFIRMED rule fires: real volume + workable position, CTR below what that position earns | `review_ctr_fix` — snippet/title rewrite | baseline score, desc |
| `high_volume_poor_position` | High prior volume, position past page 2 | `flag_for_deeper_review` — human diagnoses (technical? cannibalization? real content gap?), no auto-remedy | volume, desc |
| `model_flags_declining_no_baseline_signal` | RF probability ≥ 0.5 but the rule doesn't fire | `monitor_closely` — a watchlist, not an action queue (ML-08/09 showed the model doesn't beat the baseline at ranking) | model probability, desc |
| `thin_evidence` | Fewer than 15/15 active days in the feature window | `wait_for_more_data` — not enough measurement to trust any read | — |
| `monitor_default` | None of the above | `monitor` | — |

**A correction, made honestly:** in ML-08's error analysis I described high-volume/poor-position false positives as pages "already at a floor, nowhere lower to fall" — based on 3 examples. Checked against the full `high_volume_poor_position` archetype below (n in the thousands, not 3), the real decline rate is the **highest of all five archetypes**, not the lowest. That anecdote doesn't survive contact with the full population, so I'm dropping it and replacing it with a real number below — exactly the self-audit habit ML-09 was practicing.

In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# find the repo root from wherever this kernel started (VS Code/Colab/CLI all differ)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
con.execute("SET http_timeout=300")
con.execute("SET http_retries=5")
con.execute("SET http_retry_wait_ms=1000")
con.execute("SET http_retry_backoff=2")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

# same feature build + proxy label as ML-04/ML-07/ML-08/ML-09
feat = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_prev,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_prev,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_prev,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN report_date END) AS active_days_prev,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_last
    FROM read_parquet('{MONTH}')
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) > 0
""").df()

feat["ctr_prev"] = feat["clk_prev"] / feat["imp_prev"]
feat["is_declining"] = (feat["imp_last"] < 0.8 * feat["imp_prev"]).astype(int)
feat = feat.dropna(subset=["avg_position_prev"]).reset_index(drop=True)
feat["log_imp_prev"] = np.log1p(feat["imp_prev"])
feat["log_clk_prev"] = np.log1p(feat["clk_prev"])

FEATURES = ["log_imp_prev", "log_clk_prev", "ctr_prev", "avg_position_prev", "active_days_prev"]
print(f"shape: {feat.shape[0]:,} rows  |  clients: {feat['client_hash_id'].nunique()}  |  base decline rate: {feat['is_declining'].mean():.3f}")

shape: 150,675 rows  |  clients: 44  |  base decline rate: 0.326


In [2]:
from sklearn.ensemble import RandomForestClassifier

# 1) baseline rule (ML-07), thresholds fit on this full deployed-style run
eligible = (feat["imp_prev"] >= 100) & (feat["avg_position_prev"] > 0) & (feat["avg_position_prev"] <= 20)
tier = pd.cut(feat.loc[eligible, "avg_position_prev"], bins=[0, 3, 10, 20], labels=["1-3", "4-10", "11-20"]).astype(str)
expected_ctr_by_tier = feat.loc[eligible].assign(position_tier=tier).groupby("position_tier")["ctr_prev"].median().to_dict()

def expected_ctr_for_position(pos):
    if pos <= 3:
        return expected_ctr_by_tier["1-3"]
    if pos <= 10:
        return expected_ctr_by_tier["4-10"]
    return expected_ctr_by_tier["11-20"]

feat["expected_ctr"] = feat["avg_position_prev"].apply(expected_ctr_for_position)
gap = np.where(eligible, feat["expected_ctr"] - feat["ctr_prev"], 0.0)
feat["score_baseline"] = np.where(eligible, feat["imp_prev"] * np.clip(gap, 0, None), 0.0)

# 2) model probability (ML-08/09 recipe), fit on the full slice — descriptive scoring, not a forward claim
rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(feat[FEATURES], feat["is_declining"])
feat["model_proba"] = rf.predict_proba(feat[FEATURES])[:, 1]

# 3) archetype thresholds — real, named percentiles, not round numbers picked to fit a story
POOR_POSITION_CUT = 20  # beyond the baseline rule's own eligibility window
HIGH_VOL_CUT = feat["imp_prev"].quantile(0.90)
print(f"high-volume cutoff (90th percentile of imp_prev): {HIGH_VOL_CUT:.0f}")


def assign_archetype(row):
    if row["score_baseline"] > 0:
        return "low_ctr_for_position"
    if row["active_days_prev"] < 15:
        return "thin_evidence"
    if row["imp_prev"] >= HIGH_VOL_CUT and row["avg_position_prev"] > POOR_POSITION_CUT:
        return "high_volume_poor_position"
    if row["model_proba"] >= 0.5:
        return "model_flags_declining_no_baseline_signal"
    return "monitor_default"


feat["archetype"] = feat.apply(assign_archetype, axis=1)

ACTION_MAP = {
    "low_ctr_for_position": "review_ctr_fix",
    "high_volume_poor_position": "flag_for_deeper_review",
    "model_flags_declining_no_baseline_signal": "monitor_closely",
    "thin_evidence": "wait_for_more_data",
    "monitor_default": "monitor",
}
TIER_MAP = {  # 1 = actionable now, higher = lower priority
    "low_ctr_for_position": 1,
    "high_volume_poor_position": 2,
    "model_flags_declining_no_baseline_signal": 3,
    "thin_evidence": 4,
    "monitor_default": 5,
}
RANK_KEY = {
    "low_ctr_for_position": "score_baseline",
    "high_volume_poor_position": "imp_prev",
    "model_flags_declining_no_baseline_signal": "model_proba",
}

feat["action"] = feat["archetype"].map(ACTION_MAP)
feat["tier"] = feat["archetype"].map(TIER_MAP)
feat["rank_key"] = feat.apply(lambda r: r[RANK_KEY[r["archetype"]]] if r["archetype"] in RANK_KEY else 0.0, axis=1)

summary = feat.groupby("archetype").agg(
    n=("is_declining", "size"),
    decline_rate=("is_declining", "mean"),
    action=("action", "first"),
    tier=("tier", "first"),
).sort_values("tier")
print("\narchetype summary (real counts and decline rates, this March 2026 slice):")
summary

high-volume cutoff (90th percentile of imp_prev): 2042

archetype summary (real counts and decline rates, this March 2026 slice):


,n,decline_rate,action,tier
archetype,,,,
low_ctr_for_position,30974,0.336960,review_ctr_fix,1
high_volume_poor_position,3372,0.529359,flag_for_deeper_review,2
model_flags_declining_no_baseline_signal,7682,0.393517,monitor_closely,3
thin_evidence,76142,0.346813,wait_for_more_data,4
monitor_default,32505,0.230764,monitor,5


In [3]:
queue = feat.sort_values(["tier", "rank_key"], ascending=[True, False]).reset_index(drop=True)
queue["priority_rank"] = queue.index + 1

actionable_now = queue["tier"].isin([1, 2]).sum()
print(f"actionable-now rows (tiers 1-2): {actionable_now:,} of {len(queue):,} ({actionable_now / len(queue) * 100:.1f}%)")
print(f"watchlist (tier 3): {(queue['tier'] == 3).sum():,}")
print(f"wait / insufficient evidence (tier 4): {(queue['tier'] == 4).sum():,}")
print(f"monitor / leave alone (tier 5): {(queue['tier'] == 5).sum():,}")

show_cols = ["priority_rank", "archetype", "action", "tier", "client_hash_id", "content_hash_id",
             "imp_prev", "avg_position_prev", "ctr_prev", "active_days_prev", "model_proba", "score_baseline"]
queue[show_cols].head(10)

actionable-now rows (tiers 1-2): 34,346 of 150,675 (22.8%)
watchlist (tier 3): 7,682
wait / insufficient evidence (tier 4): 76,142
monitor / leave alone (tier 5): 32,505


,priority_rank,archetype,action,tier,client_hash_id,content_hash_id,imp_prev,avg_position_prev,ctr_prev,active_days_prev,model_proba,score_baseline
0,1,low_ctr_for_position,review_ctr_fix,1,client_62f4a7e64f5e0096,content_34a70fea29d15f24,73639.0,2.786744,0.000244,15,0.574411,177.551214
1,2,low_ctr_for_position,review_ctr_fix,1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83772.0,8.607910,0.000000,15,0.637375,152.706061
2,3,low_ctr_for_position,review_ctr_fix,1,client_62f4a7e64f5e0096,content_7c6373141eae744a,86860.0,5.785512,0.000587,15,0.627869,107.335106
3,4,low_ctr_for_position,review_ctr_fix,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,58553.0,4.579049,0.000017,15,0.605570,105.734923
4,5,low_ctr_for_position,review_ctr_fix,1,client_62f4a7e64f5e0096,content_945d6ff91386c817,49314.0,6.413782,0.000041,15,0.626299,87.893362
5,6,low_ctr_for_position,review_ctr_fix,1,client_23a62021009f63c4,content_65c75874a23fca87,55680.0,9.013531,0.000269,15,0.655623,86.497797
6,7,low_ctr_for_position,review_ctr_fix,1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,49619.0,9.493028,0.000161,15,0.629330,82.449339
7,8,low_ctr_for_position,review_ctr_fix,1,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,52378.0,4.011050,0.000344,15,0.597625,77.478657
8,9,low_ctr_for_position,review_ctr_fix,1,client_e547b89c05043229,content_306bc78dff1eb683,33020.0,1.609699,0.000424,13,0.367338,73.685888
9,10,low_ctr_for_position,review_ctr_fix,1,client_62f4a7e64f5e0096,content_36fc1ee501ec072d,46199.0,5.241369,0.000238,15,0.638768,73.215099


## 2. Intended use and limits

**Who:** a content editor or SEO lead at one client, with limited weekly review capacity (per ML-02: an average client in this slice has ~411 pages that could plausibly need attention — nobody reviews all of them by hand).

**For what:** deciding which pages to open FIRST this cycle, and why — a triage queue, not a verdict. The action column names a next step for a human to take (review a snippet, dig into why a big page fell, keep watching), never a change to publish.

**Where it stops being valid:**
- **One mid-panel month.** Everything here is built and scored on March 2026 (`month=2026-03`) only. It has not been checked against a different month, a different season, or the sealed final month — re-running this on a new month could shift the archetype thresholds and mix.
- **One proxy label.** `is_declining` is a within-month days-1-15-vs-16-31 impression ratio (verified in ML-04), not a confirmed real-world outcome across months. It's a reasonable proxy, not a certainty.
- **44 clients, one lane.** Whatever holds here hasn't been checked on clients or content types outside this slice.
- **Retrospective scoring, not a forecast.** The model and rule are fit and scored on the SAME March slice here (a "deployed-style" run) — this demonstrates the mechanics, it is not a claim about future months.
- **Decision-support only.** Nothing here is causal. "Review this page" is not "refreshing this page will recover its traffic" — that claim needs an experiment (a before/after refresh test), which this data can't provide.

In [4]:
actionable = queue[queue["tier"].isin([1, 2])]
per_client = actionable.groupby("client_hash_id").size()
print(f"clients with at least one actionable-now row: {(per_client > 0).sum()} of {feat['client_hash_id'].nunique()}")
print(f"actionable-now rows per client: mean={per_client.mean():.0f}, median={per_client.median():.0f}, "
      f"max={per_client.max()}")
print("\nEven the 'actionable now' tier is too large for a human to fully clear in one cycle at some "
      "clients — the priority_rank column inside each tier is what makes this triage rather than a to-do list.")

clients with at least one actionable-now row: 35 of 44
actionable-now rows per client: mean=981, median=250, max=8474

Even the 'actionable now' tier is too large for a human to fully clear in one cycle at some clients — the priority_rank column inside each tier is what makes this triage rather than a to-do list.


## 3. Human review + the no-go list

**What a person must check before acting on any row:**
- Read the actual page. A `low_ctr_for_position` flag says the CTR-vs-position math looks off — it doesn't know if the query is branded/navigational (structurally low CTR, nothing to fix) or if a competitor's rich snippet is stealing the click.
- Check for consolidation/seasonality/noise (the lane guide's look-alike table, ML-02) before assuming a decline is real and page-specific.
- Confirm `active_days_prev` and `imp_prev` aren't reading a tracking gap as a real pattern.

**What should NEVER be automated:**
- **Never auto-publish a rewritten title/snippet.** `review_ctr_fix` names a candidate for a human edit, not a diff to ship — an LLM-drafted snippet can misrepresent the page or clash with brand voice.
- **Never auto-prune, redirect, or delete a page from this queue.** Those actions are close to irreversible and need a content-strategy sign-off this data can't provide.
- **Never treat `model_proba` or `score_baseline` as ground truth of cause.** They rank candidates for review; ML-08/09 already showed the model doesn't outperform the rule at this ranking task, so neither number should be read as more than "worth a look."
- **Never act on a `thin_evidence` row as if it were confidently scored.** Fewer than 15/15 active days means the ratio behind `is_declining` is noisier — find out why the evidence is thin before doing anything.
- **Never surface a real client name, URL, domain, or raw query in a review report.** Everything here is pseudonym-joined; keep it that way past this notebook too.

In [5]:
# concrete privacy check: every id in the queue is still a pseudonym hash, never a raw name/url
assert queue["client_hash_id"].str.startswith("client_").all(), "found a non-pseudonym client id"
assert queue["content_hash_id"].str.startswith("content_").all(), "found a non-pseudonym content id"
assert not any(col in queue.columns for col in ["url", "domain", "query", "keyword", "client_name"]), \
    "a raw-identity column reached the queue"
print("privacy check passed: only pseudonym ids and observed metrics in the queue.")

privacy check passed: only pseudonym ids and observed metrics in the queue.


## 4. Monitoring / retrain triggers

- **Re-score at least monthly.** This queue is one month's snapshot; thresholds (`expected_ctr_by_tier`, the 90th-percentile volume cutoff) are recomputed fresh each run, not hardcoded — a new month should regenerate them, not reuse March's.
- **Precision regression trigger:** if a fresh month's `low_ctr_for_position` bucket, checked against that month's own `is_declining`, drops meaningfully below the ~0.34 decline rate confirmed here (say, toward the ~0.23 `monitor_default` rate), the rule has stopped separating risk — stop trusting it and re-audit signals (ML-07-style) before the next run.
- **Archetype-mix drift trigger:** `thin_evidence` is ~50% of rows this month (real, not a bug — see below). If that share jumps sharply in a future run (e.g., past 70%), suspect a tracking/data pipeline problem before concluding client behavior changed.
- **New-signal leakage recheck:** if a new column or FlyRank product flag is ever added as a feature, rerun the ML-09 confession test (train with/without) before trusting it.
- **Model-vs-baseline recheck:** ML-08/09 found the model doesn't beat the baseline here. If a future month's honest grouped-split comparison flips that, promote the model's probability to the primary rank key — but only after that comparison, not by assumption.

In [6]:
thin_share = (feat["archetype"] == "thin_evidence").mean()
full_coverage_share = (feat["active_days_prev"] == 15).mean()
print(f"thin_evidence share this run: {thin_share * 100:.1f}%")
print(f"rows with full 15/15-day coverage: {full_coverage_share * 100:.1f}%")
print("Lower-traffic pages naturally don't get impressions every single day — this is a real "
      "feature of a mixed-traffic portfolio, not a pipeline defect. Track this share every run "
      "as the drift baseline.")

thin_evidence share this run: 50.5%
rows with full 15/15-day coverage: 45.3%
Lower-traffic pages naturally don't get impressions every single day — this is a real feature of a mixed-traffic portfolio, not a pipeline defect. Track this share every run as the drift baseline.


## 5. Exports for the paper

Three outputs: the full ranked queue CSV (`work/outputs/` — gitignored, regenerated on every run), a metrics JSON (`work/outputs/` — committed, the receipt these numbers trace back to), and one figure (`work/figures/` — committed, the archetype correction from section 1 made visual).

In [7]:
import json
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1) the ranked queue -- gitignored, regenerated every run
queue_cols = ["priority_rank", "tier", "archetype", "action", "client_hash_id", "content_hash_id",
              "imp_prev", "clk_prev", "ctr_prev", "avg_position_prev", "active_days_prev",
              "score_baseline", "model_proba", "expected_ctr"]
queue[queue_cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)
print(f"wrote work/outputs/action_playbook_queue.csv  ({len(queue):,} rows)")

# 2) metrics JSON -- committed, the receipt
metrics = {
    "month": "2026-03",
    "n_rows": int(len(feat)),
    "n_clients": int(feat["client_hash_id"].nunique()),
    "base_decline_rate": round(float(feat["is_declining"].mean()), 4),
    "high_volume_cutoff_p90_imp_prev": round(float(HIGH_VOL_CUT), 1),
    "expected_ctr_by_position_tier": {k: round(float(v), 5) for k, v in expected_ctr_by_tier.items()},
    "archetypes": {
        name: {
            "n": int(g.shape[0]),
            "decline_rate": round(float(g["is_declining"].mean()), 4),
            "action": ACTION_MAP[name],
            "tier": TIER_MAP[name],
        }
        for name, g in feat.groupby("archetype")
    },
    "actionable_now_pct": round(float(actionable_now / len(queue) * 100), 1),
    "random_seed": 42,
}
with open("work/outputs/action_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote work/outputs/action_playbook_metrics.json")

# 3) figure -- the section-1 correction, made visual
order = summary.index.tolist()
rates = (summary["decline_rate"] * 100).values
colors = ["#c0392b" if a == "high_volume_poor_position" else "#4a4a4a" for a in order]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(order, rates, color=colors)
ax.axvline(feat["is_declining"].mean() * 100, color="black", linestyle="--", linewidth=1, label="base rate")
ax.set_xlabel("decline rate (%)")
ax.set_title("Decline rate by archetype — March 2026 slice", fontsize=12)
ax.text(0.5, -0.22, "red = the archetype that overturned my ML-08 'floor' assumption",
        transform=ax.transAxes, ha="center", fontsize=9, style="italic", color="#555555")
ax.legend()
plt.tight_layout()
plt.savefig("work/figures/archetype_decline_rates.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("wrote work/figures/archetype_decline_rates.png")

wrote work/outputs/action_playbook_queue.csv  (150,675 rows)
wrote work/outputs/action_playbook_metrics.json
wrote work/figures/archetype_decline_rates.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.